# Chapter 02 — SOLUTIONS: Data Cleaning & Preparing

**Instructor file — share only after the exercise session.**

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

sns.set_theme(style='whitegrid')
np.random.seed(0)

In [ ]:
# Same dataset setup
housing_data = {
    'house_id': range(1, 26),
    'area_sqm': [75, 120, None, 95, 200, 85, 60, 110, None, 140,
                  90, 78, 130, 9999, 105, 88, 72, 115, None, 95,
                  80, 125, 92, 67, 103],
    'rooms': [3, 4, 3, None, 5, 3, 2, 4, 3, 5,
               3, 3, 4, 4, 4, 3, 2, 4, 3, 3,
               3, 4, 3, 2, 4],
    'location': ['urban', 'suburban', 'rural', 'Urban', 'suburban',
                   'URBAN', None, 'rural', 'suburban', 'urban',
                   'Urban', 'rural', 'suburban', 'urban', None,
                   'rural', 'urban', 'suburban', 'rural', 'urban',
                   'suburban', 'urban', 'rural', 'suburban', 'urban'],
    'has_garage': ['yes', 'no', 'YES', 'Yes', 'no', 'yes', None, 'no', 'yes', 'YES',
                    'No', 'yes', 'no', 'yes', 'no', None, 'yes', 'no', 'yes', 'no',
                    'yes', 'YES', 'no', 'yes', 'no'],
    'price_1000_eur': [250, 380, 180, 290, 520, 270, 190, 360, 230, 440,
                        280, 210, 410, 275, 340, 225, 175, 370, 215, 285,
                        240, 395, 268, 165, 315]
}
df = pd.DataFrame(housing_data)
df.head()

## Task 1 Solution: Explore the Dataset

In [ ]:
# 1a: Data types
df.info()

In [ ]:
# 1b: Missing values
print('Missing values:')
print(df.isnull().sum())

In [ ]:
# 1c: Unique values
print('Unique location values:', df['location'].unique())
print('\nUnique has_garage values:', df['has_garage'].unique())
print('\nProblems: inconsistent capitalization in both columns!')

## Task 2 Solution: Fix Inconsistencies & Missing Values

In [ ]:
# 2a: Standardize location
df['location'] = df['location'].str.lower().str.strip()
print('Unique location after fix:', df['location'].unique())

In [ ]:
# 2b: Standardize has_garage and fill missing
df['has_garage'] = df['has_garage'].str.lower().str.strip()
garage_mode = df['has_garage'].mode()[0]
df['has_garage'] = df['has_garage'].fillna(garage_mode)
print('Unique has_garage after fix:', df['has_garage'].unique())
print('Mode used for fill:', garage_mode)

In [ ]:
# 2c: Fill numerical missing values with median
df['area_sqm'] = df['area_sqm'].fillna(df['area_sqm'].median())
df['rooms'] = df['rooms'].fillna(df['rooms'].median())

# Fill categorical missing values with mode
df['location'] = df['location'].fillna(df['location'].mode()[0])

print('Missing values after cleaning:')
print(df.isnull().sum())

## Task 3 Solution: Outliers

In [ ]:
# 3a: Boxplot
fig, ax = plt.subplots(figsize=(6, 4))
ax.boxplot(df['area_sqm'], patch_artist=True,
           boxprops=dict(facecolor='#3498db', alpha=0.6))
ax.set_title('Boxplot: area_sqm')
ax.set_ylabel('area_sqm')
plt.tight_layout()
plt.show()
print(f'Max value: {df["area_sqm"].max()} — clearly an error (9999 sqm!)')

In [ ]:
# 3b: IQR method
Q1 = df['area_sqm'].quantile(0.25)
Q3 = df['area_sqm'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 1.5 * IQR

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}, upper_bound={upper_bound:.1f}')

area_median = df.loc[df['area_sqm'] <= upper_bound, 'area_sqm'].median()
df.loc[df['area_sqm'] > upper_bound, 'area_sqm'] = area_median

print(f'Max area_sqm after fix: {df["area_sqm"].max()}')

## Task 4 Solution: Encoding

In [ ]:
# 4a: Binary encoding for has_garage
df['garage_encoded'] = (df['has_garage'] == 'yes').astype(int)
print('Garage encoding sample:')
print(df[['has_garage', 'garage_encoded']].head(5))

In [ ]:
# 4b: One-hot encoding for location
location_dummies = pd.get_dummies(df['location'], prefix='loc', dtype=int)
df = pd.concat([df, location_dummies], axis=1)

print('Columns after encoding:')
print(df.columns.tolist())
df[['location', 'loc_rural', 'loc_suburban', 'loc_urban']].head()

## Task 5 Solution: Scale & Split

In [ ]:
feature_cols = ['area_sqm', 'rooms', 'garage_encoded', 'loc_rural', 'loc_suburban', 'loc_urban']
target_col = 'price_1000_eur'

X = df[feature_cols].copy()
y = df[target_col]

# 5a: Train/test split FIRST
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
print('Train size:', X_train.shape[0], '| Test size:', X_test.shape[0])

In [ ]:
# 5b: Scale AFTER the split (only fit on train!)
scaler = StandardScaler()
numerical_cols = ['area_sqm', 'rooms']

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])  # transform only, not fit!

print('X_train (scaled) sample:')
print(X_train.head())

print(f'\n✅ Data cleaned and ready!')
print(f'   Training: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

## Bonus Solution: Correlation Heatmap

In [ ]:
corr_cols = ['area_sqm', 'rooms', 'garage_encoded', 'price_1000_eur']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title('Correlation with House Price', fontsize=13)
plt.tight_layout()
plt.show()

print('\nConclusion: area_sqm and rooms are most correlated with price.')
print('Makes intuitive sense — bigger houses cost more!')